In [5]:
import openai
import time
import re
import pandas as pd

def send_message(
    message, 
    max_tokens=1000,          # Увеличил дефолт, так как таблицы и логи могут быть длинными
    top_p=0.9, 
    temperature=0.0,          # Для задач интерпретатора строго 0.0 по умолчанию
    server_url="http://127.0.0.1:9092/v1", 
    api_key="dummy",
    model_name='Qwen2.5-Coder-7B-Instruct', 
    stop=None,                # Для обычных ответов стоп-слова лучше сделать опциональными
    retries=3                 # Количество попыток при падении сервера
):
    # Инициализируем клиент OpenAI
    client = openai.OpenAI(base_url=server_url, api_key=api_key)
    
    model_input = [
        { 'role': 'user', 'content': message}
    ]
    
    try:
        print(f"Generating content with model: {model_name} (Temp: {temperature})")
        
        response = client.chat.completions.create(
            model=model_name,
            messages=model_input,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stop=stop
        )
        
        return True, response.choices[0].message.content

    except Exception as e:
        print(f"\n[ERROR] Failed to call LLM: {e}")
        
        # Разбираем ответ сервера, если он есть
        if hasattr(e, 'response') and e.response is not None:
            try:
                error_info = e.response.json()  
                code_value = error_info.get('error', {}).get('code', 'unknown_error')
                print(f"Код ошибки от сервера: {code_value}")
            except Exception:
                print(f"Сырой ответ сервера об ошибке: {e.response.text}")
        
        # Если попытки еще остались — пробуем снова
        if retries > 0:
            print(f"Waiting 6 seconds before retry... (Remaining retries: {retries})")
            time.sleep(6)
            return send_message(
                message=message, max_tokens=max_tokens, top_p=top_p, 
                temperature=temperature, server_url=server_url, api_key=api_key, 
                model_name=model_name, stop=stop, retries=retries-1
            )
        else:
            print("All retries failed. Skipping.")
            return False, None

def parse_panda_code(input_string):
    # Сначала попробуем найти JSON объект с PANDA
    json_pattern = r'\{[^{}]*(?:CORRECT PANDA|PANDA)":\s*(.+?)(?:\n|$)?\}'
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    code = None
    pattern = r'"(?:CORRECT PANDA|PANDA)":\s*(.+?)(?:\n|$)'
    if json_match:
        code = json_match.group(1).strip()
    else:
        match = re.search(pattern, input_string, re.DOTALL)
        if match:
            code = match.group(1).strip()

    if code != None:
        if code.startswith('"') and code.endswith('"'):
            code = code[1:-1]
        elif code.startswith("'") and code.endswith("'"):
            code = code[1:-1]
            
        return code
    
    return ""


In [13]:
train = pd.read_csv('data/data/training.tsv', sep = '\t')
train

,id,utterance,context,targetValue
0,nt-0,what was the last year where this team was a p...,csv/204-csv/590.csv,2004
1,nt-1,in what city did piotr's last 1st place finish...,csv/204-csv/622.csv,"Bangkok, Thailand"
2,nt-2,which team won previous to crettyard?,csv/204-csv/772.csv,Wolfe Tones
3,nt-3,how many more passengers flew to los angeles t...,csv/203-csv/515.csv,"12,467"
4,nt-4,who was the opponent in the first game of the ...,csv/204-csv/495.csv,Derby County
...,...,...,...,...
14144,nt-14147,who came in last?,csv/204-csv/433.csv,Javier Díaz
14145,nt-14148,which album has the highest number of sales bu...,csv/204-csv/949.csv,Vain elämää
14146,nt-14149,japan finished below how many countries?,csv/204-csv/183.csv,0
14147,nt-14150,how many districts have a population density o...,csv/204-csv/739.csv,31


In [7]:
csv = pd.read_csv('data/'+ train.context[0])
csv

,Year,Division,League,Regular Season,Playoffs,Open Cup,Avg. Attendance
0,2001,2,USL A-League,"4th, Western",Quarterfinals,Did not qualify,"7,169"
1,2002,2,USL A-League,"2nd, Pacific",1st Round,Did not qualify,"6,260"
2,2003,2,USL A-League,"3rd, Pacific",Did not qualify,Did not qualify,"5,871"
3,2004,2,USL A-League,"1st, Western",Quarterfinals,4th Round,"5,628"
4,2005,2,USL First Division,5th,Quarterfinals,4th Round,"6,028"
5,2006,2,USL First Division,11th,Did not qualify,3rd Round,"5,575"
6,2007,2,USL First Division,2nd,Semifinals,2nd Round,"6,851"
7,2008,2,USL First Division,11th,Did not qualify,1st Round,"8,567"
8,2009,2,USL First Division,1st,Semifinals,3rd Round,"9,734"
9,2010,2,USSF D-2 Pro League,"3rd, USL (3rd)",Quarterfinals,3rd Round,"10,727"


In [124]:
df_table = pd.read_csv('data/' + train.context.iloc[i])
print(df_table)

   Year  Division               League  Regular Season         Playoffs  \
0  2001         2         USL A-League    4th, Western    Quarterfinals   
1  2002         2         USL A-League    2nd, Pacific        1st Round   
2  2003         2         USL A-League    3rd, Pacific  Did not qualify   
3  2004         2         USL A-League    1st, Western    Quarterfinals   
4  2005         2   USL First Division             5th    Quarterfinals   
5  2006         2   USL First Division            11th  Did not qualify   
6  2007         2   USL First Division             2nd       Semifinals   
7  2008         2   USL First Division            11th  Did not qualify   
8  2009         2   USL First Division             1st       Semifinals   
9  2010         2  USSF D-2 Pro League  3rd, USL (3rd)    Quarterfinals   

          Open Cup Avg. Attendance  
0  Did not qualify           7,169  
1  Did not qualify           6,260  
2  Did not qualify           5,871  
3        4th Round        

In [19]:
import config
import importlib
import pandas as pd

# 0. Принудительно перезагружаем конфиг
importlib.reload(config) 

# Предположим, мы берем первый пример (i = 0)
i = 0

# ИСПРАВЛЕНИЕ: Убедитесь, что переменная `train` у вас определена выше в коде (например, train = pd.read_csv('...'))

# 1. Достаем чистый текст вопроса
query_text = train.utterance.iloc[i]

# 2. Читаем таблицу и превращаем ЕЁ В ТЕКСТ (в формат Markdown)
table_text = pd.read_csv('data/' + train.context.iloc[i]).to_string(index=False)
#table_text = df_table.to_markdown(index=False) 

# 3. Безопасная подстановка данных через .replace() вместо .format()
# Это защитит от ошибок, если в промпте есть другие фигурные скобки {}
full_message = config.system_prompt.replace("{table}", table_text).replace("{query}", query_text)

print("Отправляем корректный запрос на локальный Qwen...")

# ИСПРАВЛЕНИЕ: Передаем именно full_message
success, response = send_message(
    message=full_message,
)

    
print(success, response)
if success:
    print("\n--- Ответ от модели ---")
    print(response)
    print(full_message)
    parsed_code = parse_panda_code(response)
    print("\n--- Извлеченный код Pandas ---")
    print(f"[{parsed_code}]")
else:
    print("Не удалось получить ответ от сервера.")

Отправляем корректный запрос на локальный Qwen...
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
True ```json
{
  "PANDA": "df.loc[df['Division'] == 2 & df['League'] == 'USL A-League', 'Year'].max()"
}
```

--- Ответ от модели ---
```json
{
  "PANDA": "df.loc[df['Division'] == 2 & df['League'] == 'USL A-League', 'Year'].max()"
}
```
You are a Python expert specializing in pandas. You are given a question and a table. Your task is to translate the given natural language question into
a single-line pandas expression. This expression, which acts like a query, must
be valid and executable so that running the pandas expression will output the
answer to the question. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run, it outputs the correct
given answer, and strictly follows the Json format: {{"PANDA": "<your Pandas code

In [57]:
import json
import re

def exec_pandas(json_str):
    # Удаляем markdown-обертку ```json и ```
    cleaned = re.sub(r'^```json\n|\n```$', '', json_str.strip())
    return json.loads(cleaned)["PANDA"]

In [67]:
import importlib
import config
importlib.reload(config) 

n = 10

code = {
    'correct': '',
    're_correct': '',
    're_uncorrect': ''   # исправлена опечатка
}

for i in range(n):
    query_text = train.utterance.iloc[i]
    table_path = 'data/' + train.context.iloc[i]
    table_text = pd.read_csv(table_path).to_string(index=False)
    full_message = config.system_prompt.replace("{table}", table_text).replace("{query}", query_text)

    success, response = send_message(message=full_message)
    
    if not success:
        continue  # или обработка ошибки

    label = train.targetValue.iloc[i]  # вынести сюда, чтобы была доступна везде

    try:
        df = pd.read_csv(table_path)
        result = eval(exec_pandas(response))
        if result == label:
            code['correct'] += " " + exec_pandas(response)
        else:
            logic_message = config.logic_prompt.replace("{table}", table_text).replace("{query}", query_text).replace("{label}", str(label)).replace("{pandas}", exec_pandas(response))
            success2, response2 = send_message(message=logic_message)
            if success2:
                try:
                    if eval(exec_pandas(response2)) == label:
                        code["re_correct"] += " " + exec_pandas(response2)
                except:
                    code["re_uncorrect"] += " " + exec_pandas(response2)
    except Exception as e:
        # label уже определена
        correct_message = config.correct_prompt.replace("{table}", table_text).replace("{query}", query_text).replace("{label}", str(label)).replace("{pandas}", exec_pandas(response))
        success2, response2 = send_message(message=correct_message)
        if success2:
            try:
                if eval(exec_pandas(response2)) == label:
                    code["re_correct"] += " " + exec_pandas(response2)
            except:
                code["re_uncorrect"] += " " + exec_pandas(response2)

Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwe

In [71]:
correct_count = 0
logic_count = 0
syntax_count = 0
failed_count = 0

n = 10

code = {
    'correct': '',
    're_correct': '',
    're_uncorrect': ''
}

for i in range(n):
    query_text = train.utterance.iloc[i]
    table_path = 'data/' + train.context.iloc[i]
    table_text = pd.read_csv(table_path).to_string(index=False)
    full_message = config.system_prompt.replace("{table}", table_text).replace("{query}", query_text)

    success, response = send_message(message=full_message)
    
    if not success:
        failed_count += 1
        print(f"[{i}] send_message failed")
        continue

    label = train.targetValue.iloc[i]
    pandas_code = exec_pandas(response)
    
    if not pandas_code:
        failed_count += 1
        print(f"[{i}] No pandas code extracted")
        continue

    # Первая попытка: выполнить сгенерированный код
    try:
        df = pd.read_csv(table_path)
        result = eval(pandas_code)
        
        if result == label:
            code['correct'] += " " + pandas_code
            correct_count += 1
        else:
            # Логическая коррекция
            logic_message = config.logic_prompt.replace("{table}", table_text).replace("{query}", query_text).replace("{label}", str(label)).replace("{pandas}", pandas_code)
            success2, response2 = send_message(message=logic_message)
            
            if success2:
                pandas_code2 = exec_pandas(response2)
                try:
                    if eval(pandas_code2) == label:
                        code["re_correct"] += " " + pandas_code2
                        logic_count += 1
                    else:
                        code["re_uncorrect"] += " " + pandas_code2
                        failed_count += 1
                except:
                    code["re_uncorrect"] += " " + pandas_code2
                    failed_count += 1
            else:
                failed_count += 1
                
    except Exception as e:
        # Синтаксическая коррекция (код не выполнился из-за ошибки)
        print(f"[{i}] Execution error: {e}")
        
        syntax_message = config.correct_prompt.replace("{table}", table_text).replace("{query}", query_text).replace("{label}", str(label)).replace("{pandas}", pandas_code)
        success2, response2 = send_message(message=syntax_message)
        
        if success2:
            pandas_code2 = exec_pandas(response2)
            try:
                if eval(pandas_code2) == label:
                    code["re_correct"] += " " + pandas_code2
                    syntax_count += 1
                else:
                    code["re_uncorrect"] += " " + pandas_code2
                    failed_count += 1
            except:
                code["re_uncorrect"] += " " + pandas_code2
                failed_count += 1
        else:
            failed_count += 1

print(f"Correct: {correct_count}, Logic: {logic_count}, Syntax: {syntax_count}, Failed: {failed_count}")
print(f"Total processed: {correct_count + logic_count + syntax_count + failed_count}")

Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
[0] Execution error: operation 'rand_' not supported for dtype 'str' with object of type <class 'int'>
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
[3] Execution error: unsupported operand type(s) for -: 'str' and 'str'
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
[5] Execution error: operation 'sub' not supported for dtype 'str' with dtype 'datetime64[us]'
Generating content with model: Qwen2.5-Coder-7B-Instruct (Temp: 0.0)
Generating content with model: Qwen2.5-C

In [73]:
code

{'correct': " df.loc[df['Position'] == '1st', 'Venue'].iloc[-1] df['Opponent'].iloc[0]",
 're_correct': '',
 're_uncorrect': " df.loc[(df['Division'] == 2) & (df['League'] == 'USL A-League'), 'Year'].max() df.loc[df['Team'] == 'Wolfe Tones', 'Years won'].iloc[0] - 1 (df.loc[df['City'] == 'United States, Los Angeles', 'Passengers'].iloc[0] - df.loc[df['City'] == 'Canada, Saskatoon', 'Passengers'].iloc[0]) len(df[(df['Left office'].dt.to_period('D') - df['Took office'].dt.to_period('D')).astype(int') >= 1095]) df['Away team'].iloc[0] df.loc[df['Name in English'] == 'Lake Palas Tuzla', 'Depth'].values[0] df.loc['Full house', '4 credits'] df[(df['Position'] == df.loc[3, 'Position']) & (df['Player'] != 'Siim Ennemuist')]['Player'].tolist()"}

In [59]:
for i in range(len(code)):
    current code = exec_pandas(code[i])
    df = pd.read_csv('data/' + train.context.iloc[i])
    target = train.targetValue.iloc[i]

df.loc[df['Division'] == 2 & df['League'] == 'USL A-League', 'Year'].max()
df.loc[df['Position'] == '1st', 'Venue'].iloc[-1]
df.loc[df['Team'] == 'Crettyard', 'Years won'].iloc[0] - 1
(df.loc[df['City'] == 'United States, Los Angeles', 'Passengers'].values[0] - df.loc[df['City'] == 'Canada, Saskatoon', 'Passengers'].values[0])
df['Opponent'].iloc[0]
len(df[df['Left office'] - pd.to_datetime(df['Took office']) >= pd.Timedelta(days=1095)])
df['Away team'].iloc[0]
df.loc[df['Name in English'].isin(['Lake Tuz', 'Lake Palas Tuzla']), 'Depth'].max()
df.loc['Full house', '4 credits']
df[df['Position'] == df.loc[3, 'Position']]['Player'].tolist()


In [55]:
code

{0: '```json\n{\n  "PANDA": "df.loc[df[\'Division\'] == 2 & df[\'League\'] == \'USL A-League\', \'Year\'].max()"\n}\n```',
 1: '```json\n{\n  "PANDA": "df.loc[df[\'Position\'] == \'1st\', \'Venue\'].iloc[-1]"\n}\n```',
 2: '```json\n{\n  "PANDA": "df.loc[df[\'Team\'] == \'Crettyard\', \'Years won\'].iloc[0] - 1"\n}\n```',
 3: '```json\n{\n  "PANDA": "(df.loc[df[\'City\'] == \'United States, Los Angeles\', \'Passengers\'].values[0] - df.loc[df[\'City\'] == \'Canada, Saskatoon\', \'Passengers\'].values[0])"\n}\n```',
 4: '```json\n{\n  "PANDA": "df[\'Opponent\'].iloc[0]"\n}\n```',
 5: '```json\n{\n  "PANDA": "len(df[df[\'Left office\'] - pd.to_datetime(df[\'Took office\']) >= pd.Timedelta(days=1095)])"\n}\n```',
 6: '```json\n{\n  "PANDA": "df[\'Away team\'].iloc[0]"\n}\n```',
 7: '```json\n{\n  "PANDA": "df.loc[df[\'Name in English\'].isin([\'Lake Tuz\', \'Lake Palas Tuzla\']), \'Depth\'].max()"\n}\n```',
 8: '```json\n{\n  "PANDA": "df.loc[\'Full house\', \'4 credits\']"\n}\n```',
 9: 

In [156]:
for i in range(n):
    print(code[i])
    print(pd.read_csv('data/' + train.context.iloc[i]).to_string(index=False))
    print(train.utterance.iloc[i])

```json
{"PANDA": "df[df['League'] == 'USL A-League'].index[-1].year"}
```
 Year  Division              League Regular Season        Playoffs        Open Cup Avg. Attendance
 2001         2        USL A-League   4th, Western   Quarterfinals Did not qualify           7,169
 2002         2        USL A-League   2nd, Pacific       1st Round Did not qualify           6,260
 2003         2        USL A-League   3rd, Pacific Did not qualify Did not qualify           5,871
 2004         2        USL A-League   1st, Western   Quarterfinals       4th Round           5,628
 2005         2  USL First Division            5th   Quarterfinals       4th Round           6,028
 2006         2  USL First Division           11th Did not qualify       3rd Round           5,575
 2007         2  USL First Division            2nd      Semifinals       2nd Round           6,851
 2008         2  USL First Division           11th Did not qualify       1st Round           8,567
 2009         2  USL First Divisio

In [87]:
for i in range(n):
    # Очищаем код для текущей итерации
    current_code = parse_panda_code(code[i])
    
    #print(f"Итерация {i} | Запускаем выражение: {current_code}")
    
    try:
        # Читаем нужный датафрейм
        df = pd.read_csv('data/' + train.context.iloc[i])
        target = train.iloc[i].targetValue
        
        # Передаем в eval() код ИМЕННО для текущей итерации
        result = eval(current_code)
        
        # print("--- Результат ---")
        # print(result)
        # print("--- Правильный ответ ---")
        # print(target)
        # print("-" * 40)
        if result == target:
            print(current_code)
        
    except Exception as e:
        print()
        # Выводим реальный текст ошибки, чтобы понять, в чем проблема
        #print(f"Ошибка на итерации {i}: {e}")
        #print("-" * 40)


df[df['Position'] == '1st']['Venue'].iloc[-1]


df['Opponent'][0]






In [61]:
df = pd.read_csv('data/'+ train.context.iloc[0])

In [75]:
! pip freeze > requirements.txt